<a href="https://colab.research.google.com/github/serahnjogu-new/Climate-and-health-risk-prediction/blob/main/climate_x_health_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
nasa_power_download.py
======================
Downloads daily NASA POWER weather data for every unique (latitude, longitude)
in Train/Test, then builds per-row rolling-window features.

WHAT YOU NEED
    pip install pandas numpy requests
    Folder layout (run the script from the folder that contains Data/):
        Data/Train.csv
        Data/Test.csv

HOW TO RUN
    python nasa_power_download.py
    (optional)  python nasa_power_download.py --data-dir Data --out-dir nasa_output

OUTPUT
    nasa_output/nasa_features.csv   one row per ID (~2 MB)  <-- UPLOAD THIS ONE to Claude
    nasa_output/nasa_daily.csv      raw daily values (~50 MB) - keep it, upload only if asked

NOTES
    - Free public API, no account / API key needed.
    - 55 locations x 16 years = ~880 requests, roughly 30-50 minutes. It is RESUMABLE: every location-year is cached
      as a JSON file in nasa_output/cache/, so if it stops, just run it again.
    - Source to disclose on the Zindi forum: NASA POWER (https://power.larc.nasa.gov),
      daily point API, community=AG.
"""
import argparse
import json
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

# T2M temp | T2M_MAX / T2M_MIN | PRECTOTCORR rain | RH2M humidity |
# WS2M wind speed | ALLSKY_SFC_SW_DWN solar radiation
PARAMETERS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "ALLSKY_SFC_SW_DWN"]
FILL_VALUE = -999.0
WINDOWS = [7, 30, 90]          # days BEFORE the death date
REQUEST_PAUSE_SEC = 1.2        # be polite to the free API
MAX_RETRIES = 5


# --------------------------------------------------------------------------
# 1. Inputs
# --------------------------------------------------------------------------
def load_inputs(data_dir: Path):
    train = pd.read_csv(data_dir / "Train (6).csv", parse_dates=["deathdate"])
    test = pd.read_csv(data_dir / "Test (7).csv", parse_dates=["deathdate"])
    for df in (train, test):
        df["latitude"] = df["latitude"].round(6)
        df["longitude"] = df["longitude"].round(6)
    return train, test


def get_locations_and_dates(train: pd.DataFrame, test: pd.DataFrame):
    locs = (
        pd.concat([train[["latitude", "longitude"]], test[["latitude", "longitude"]]])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    all_dates = pd.concat([train["deathdate"], test["deathdate"]])
    # 95 days of history so the 90-day window is complete for the earliest death
    start = (all_dates.min() - pd.Timedelta(days=95)).date()
    end = all_dates.max().date()
    return locs, start, end


# --------------------------------------------------------------------------
# 2. Download (with cache + retries)
# --------------------------------------------------------------------------
def cache_file(cache_dir: Path, lat: float, lon: float, year: int) -> Path:
    return cache_dir / f"{lat:.6f}_{lon:.6f}_{year}.json"


def fetch_chunk(session: requests.Session, lat: float, lon: float, start: date, end: date):
    """One API call for one location and one date range. Returns parsed JSON or None."""
    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start.strftime("%Y%m%d"),
        "end": end.strftime("%Y%m%d"),
        "format": "JSON",
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(API_URL, params=params, timeout=90)
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = 5 * attempt
                print(f"    HTTP {resp.status_code}; retry {attempt}/{MAX_RETRIES} in {wait}s", flush=True)
                time.sleep(wait)
                continue
            print(f"    HTTP {resp.status_code} (not retrying): {resp.text[:200]}", flush=True)
            return None
        except (requests.RequestException, ValueError) as exc:
            wait = 5 * attempt
            print(f"    {type(exc).__name__}; retry {attempt}/{MAX_RETRIES} in {wait}s", flush=True)
            time.sleep(wait)
    return None


def download_all(locs, start: date, end: date, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    years = range(start.year, end.year + 1)
    total_chunks = len(locs) * len(years)
    done = failed = 0

    for i, row in locs.iterrows():
        lat, lon = float(row["latitude"]), float(row["longitude"])
        for year in years:
            path = cache_file(cache_dir, lat, lon, year)
            if path.exists() and path.stat().st_size > 0:
                done += 1
                continue
            chunk_start = max(start, date(year, 1, 1))
            chunk_end = min(end, date(year, 12, 31))
            payload = fetch_chunk(session, lat, lon, chunk_start, chunk_end)
            if payload is None or "properties" not in payload:
                failed += 1
                continue
            path.write_text(json.dumps(payload))
            done += 1
            time.sleep(REQUEST_PAUSE_SEC)
        print(f"[{i + 1}/{len(locs)}] ({lat:.4f}, {lon:.4f})  chunks ok={done}/{total_chunks}  failed={failed}", flush=True)

    if failed:
        print(f"\n{failed} chunk(s) failed. Just run the script again - finished chunks are cached "
              f"and only the failed ones will be retried.", flush=True)
    return failed


# --------------------------------------------------------------------------
# 3. Parse + consolidate
# --------------------------------------------------------------------------
def parse_payload(payload: dict, lat: float, lon: float) -> pd.DataFrame:
    """Turn one API response into a tidy daily DataFrame (fill values -> NaN)."""
    params = payload["properties"]["parameter"]
    frame = pd.DataFrame({name: pd.Series(values) for name, values in params.items()})
    frame.index = pd.to_datetime(frame.index, format="%Y%m%d")
    frame.index.name = "date"
    frame = frame.replace(FILL_VALUE, np.nan).reset_index()
    frame.insert(1, "latitude", lat)
    frame.insert(2, "longitude", lon)
    return frame


def consolidate(locs, start: date, end: date, cache_dir: Path) -> pd.DataFrame:
    frames = []
    for _, row in locs.iterrows():
        lat, lon = float(row["latitude"]), float(row["longitude"])
        for year in range(start.year, end.year + 1):
            path = cache_file(cache_dir, lat, lon, year)
            if not path.exists() or path.stat().st_size == 0:
                continue
            try:
                frames.append(parse_payload(json.loads(path.read_text()), lat, lon))
            except (KeyError, ValueError):
                print(f"  skipping unreadable cache file: {path.name}", flush=True)
    if not frames:
        raise RuntimeError("No NASA data could be read. Check your internet connection and rerun.")
    daily = pd.concat(frames, ignore_index=True)
    daily["latitude"] = daily["latitude"].round(6)
    daily["longitude"] = daily["longitude"].round(6)
    return daily.sort_values(["latitude", "longitude", "date"]).reset_index(drop=True)


# --------------------------------------------------------------------------
# 4. Per-ID rolling features (windows end the day BEFORE the death date)
# --------------------------------------------------------------------------
def build_id_features(daily: pd.DataFrame, train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    rolled = []
    for (lat, lon), grp in daily.groupby(["latitude", "longitude"]):
        grp = grp.sort_values("date").set_index("date")
        # reindex to a complete daily calendar so "7 days" always means 7 calendar days
        grp = grp.reindex(pd.date_range(grp.index.min(), grp.index.max(), freq="D"))
        out = pd.DataFrame(index=grp.index)
        for var in PARAMETERS:
            series = grp[var].shift(1)    # exclude the death date itself
            for w in WINDOWS:
                out[f"nasa_{var}_mean_{w}d"] = series.rolling(w, min_periods=max(1, w // 2)).mean()
            out[f"nasa_{var}_std_30d"] = series.rolling(30, min_periods=15).std()
        # accumulated rain over each window
        rain = grp["PRECTOTCORR"].shift(1)
        for w in WINDOWS:
            out[f"nasa_rain_sum_{w}d"] = rain.rolling(w, min_periods=max(1, w // 2)).sum()
        out["latitude"], out["longitude"] = lat, lon
        out.index.name = "deathdate"
        rolled.append(out.reset_index())
    lookup = pd.concat(rolled, ignore_index=True)

    people = pd.concat([train[["ID", "latitude", "longitude", "deathdate"]],
                        test[["ID", "latitude", "longitude", "deathdate"]]], ignore_index=True)
    people["deathdate"] = pd.to_datetime(people["deathdate"]).dt.normalize()
    features = people.merge(lookup, on=["latitude", "longitude", "deathdate"], how="left")
    return features.drop(columns=["latitude", "longitude", "deathdate"])


# --------------------------------------------------------------------------
# 5. Main
# --------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="Data", help="folder containing Train.csv and Test.csv")
    parser.add_argument("--out-dir", default="nasa_output", help="where results are written")
    args = parser.parse_args()

    data_dir, out_dir = Path(args.data_dir), Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train, test = load_inputs(data_dir)
    locs, start, end = get_locations_and_dates(train, test)
    print(f"{len(locs)} unique locations | {start} -> {end} | "
          f"{len(locs) * (end.year - start.year + 1)} requests max\n", flush=True)

    failed = download_all(locs, start, end, out_dir / "cache")

    daily = consolidate(locs, start, end, out_dir / "cache")
    daily.to_csv(out_dir / "nasa_daily.csv", index=False)
    print(f"\nSaved nasa_daily.csv: {len(daily):,} rows", flush=True)

    features = build_id_features(daily, train, test)
    features.to_csv(out_dir / "nasa_features.csv", index=False)
    coverage = features.filter(like="nasa_").notna().all(axis=1).mean()
    print(f"Saved nasa_features.csv: {features.shape[0]:,} rows x {features.shape[1] - 1} features "
          f"| rows with complete features: {coverage:.1%}", flush=True)

    if failed or coverage < 0.95:
        print("\nSome data is missing - rerun the script to retry failed chunks before uploading.")
        sys.exit(1)
    print("\nAll done. Upload nasa_output/nasa_features.csv back to Claude.")


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--data-dir DATA_DIR] [--out-dir OUT_DIR]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-3f1fc482-236f-4155-8890-92dd845ed35e.json


SystemExit: 2

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
!mkdir -p Data
!cp "Train (6).csv" Data/Train.csv
!cp "Test (7).csv" Data/Test.csv
!ls Data

Test.csv  Train.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
"""
nasa_power_download.py
======================
Downloads daily NASA POWER weather data for every unique (latitude, longitude)
in Train/Test, then builds per-row rolling-window features.

WHAT YOU NEED
    pip install pandas numpy requests
    Folder layout (run the script from the folder that contains Data/):
        Data/Train.csv
        Data/Test.csv

HOW TO RUN
    Terminal:   python nasa_power_download.py
    Colab:      upload this file + your Data/ folder, then in a cell run
                    !python nasa_power_download.py
                (to keep the cache if Colab disconnects, mount Google Drive and use
                 --out-dir /content/drive/MyDrive/nasa_output)
    (optional)  python nasa_power_download.py --data-dir Data --out-dir nasa_output

OUTPUT
    nasa_output/nasa_features.csv   one row per ID (~2 MB)  <-- UPLOAD THIS ONE to Claude
    nasa_output/nasa_daily.csv      raw daily values (~50 MB) - keep it, upload only if asked

NOTES
    - Free public API, no account / API key needed.
    - 55 locations x 16 years = ~880 requests, roughly 30-50 minutes. It is RESUMABLE: every location-year is cached
      as a JSON file in nasa_output/cache/, so if it stops, just run it again.
    - Source to disclose on the Zindi forum: NASA POWER (https://power.larc.nasa.gov),
      daily point API, community=AG.
"""
import argparse
import json
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import requests

API_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

# T2M temp | T2M_MAX / T2M_MIN | PRECTOTCORR rain | RH2M humidity |
# WS2M wind speed | ALLSKY_SFC_SW_DWN solar radiation
PARAMETERS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "ALLSKY_SFC_SW_DWN"]
FILL_VALUE = -999.0
WINDOWS = [7, 30, 90]          # days BEFORE the death date
REQUEST_PAUSE_SEC = 1.2        # be polite to the free API
MAX_RETRIES = 5


# --------------------------------------------------------------------------
# 1. Inputs
# --------------------------------------------------------------------------
def load_inputs(data_dir: Path):
    train = pd.read_csv(data_dir / "Train.csv", parse_dates=["deathdate"])
    test = pd.read_csv(data_dir / "Test.csv", parse_dates=["deathdate"])
    for df in (train, test):
        df["latitude"] = df["latitude"].round(6)
        df["longitude"] = df["longitude"].round(6)
    return train, test


def get_locations_and_dates(train: pd.DataFrame, test: pd.DataFrame):
    locs = (
        pd.concat([train[["latitude", "longitude"]], test[["latitude", "longitude"]]])
        .drop_duplicates()
        .reset_index(drop=True)
    )
    all_dates = pd.concat([train["deathdate"], test["deathdate"]])
    # 95 days of history so the 90-day window is complete for the earliest death
    start = (all_dates.min() - pd.Timedelta(days=95)).date()
    end = all_dates.max().date()
    return locs, start, end


# --------------------------------------------------------------------------
# 2. Download (with cache + retries)
# --------------------------------------------------------------------------
def cache_file(cache_dir: Path, lat: float, lon: float, year: int) -> Path:
    return cache_dir / f"{lat:.6f}_{lon:.6f}_{year}.json"


def fetch_chunk(session: requests.Session, lat: float, lon: float, start: date, end: date):
    """One API call for one location and one date range. Returns parsed JSON or None."""
    params = {
        "parameters": ",".join(PARAMETERS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start.strftime("%Y%m%d"),
        "end": end.strftime("%Y%m%d"),
        "format": "JSON",
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(API_URL, params=params, timeout=90)
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code in (429, 500, 502, 503, 504):
                wait = 5 * attempt
                print(f"    HTTP {resp.status_code}; retry {attempt}/{MAX_RETRIES} in {wait}s", flush=True)
                time.sleep(wait)
                continue
            print(f"    HTTP {resp.status_code} (not retrying): {resp.text[:200]}", flush=True)
            return None
        except (requests.RequestException, ValueError) as exc:
            wait = 5 * attempt
            print(f"    {type(exc).__name__}; retry {attempt}/{MAX_RETRIES} in {wait}s", flush=True)
            time.sleep(wait)
    return None


def download_all(locs, start: date, end: date, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    session = requests.Session()
    years = range(start.year, end.year + 1)
    total_chunks = len(locs) * len(years)
    done = failed = 0

    for i, row in locs.iterrows():
        lat, lon = float(row["latitude"]), float(row["longitude"])
        for year in years:
            path = cache_file(cache_dir, lat, lon, year)
            if path.exists() and path.stat().st_size > 0:
                done += 1
                continue
            chunk_start = max(start, date(year, 1, 1))
            chunk_end = min(end, date(year, 12, 31))
            payload = fetch_chunk(session, lat, lon, chunk_start, chunk_end)
            if payload is None or "properties" not in payload:
                failed += 1
                continue
            path.write_text(json.dumps(payload))
            done += 1
            time.sleep(REQUEST_PAUSE_SEC)
        print(f"[{i + 1}/{len(locs)}] ({lat:.4f}, {lon:.4f})  chunks ok={done}/{total_chunks}  failed={failed}", flush=True)

    if failed:
        print(f"\n{failed} chunk(s) failed. Just run the script again - finished chunks are cached "
              f"and only the failed ones will be retried.", flush=True)
    return failed


# --------------------------------------------------------------------------
# 3. Parse + consolidate
# --------------------------------------------------------------------------
def parse_payload(payload: dict, lat: float, lon: float) -> pd.DataFrame:
    """Turn one API response into a tidy daily DataFrame (fill values -> NaN)."""
    params = payload["properties"]["parameter"]
    frame = pd.DataFrame({name: pd.Series(values) for name, values in params.items()})
    frame.index = pd.to_datetime(frame.index, format="%Y%m%d")
    frame.index.name = "date"
    frame = frame.replace(FILL_VALUE, np.nan).reset_index()
    frame.insert(1, "latitude", lat)
    frame.insert(2, "longitude", lon)
    return frame


def consolidate(locs, start: date, end: date, cache_dir: Path) -> pd.DataFrame:
    frames = []
    for _, row in locs.iterrows():
        lat, lon = float(row["latitude"]), float(row["longitude"])
        for year in range(start.year, end.year + 1):
            path = cache_file(cache_dir, lat, lon, year)
            if not path.exists() or path.stat().st_size == 0:
                continue
            try:
                frames.append(parse_payload(json.loads(path.read_text()), lat, lon))
            except (KeyError, ValueError):
                print(f"  skipping unreadable cache file: {path.name}", flush=True)
    if not frames:
        raise RuntimeError("No NASA data could be read. Check your internet connection and rerun.")
    daily = pd.concat(frames, ignore_index=True)
    daily["latitude"] = daily["latitude"].round(6)
    daily["longitude"] = daily["longitude"].round(6)
    return daily.sort_values(["latitude", "longitude", "date"]).reset_index(drop=True)


# --------------------------------------------------------------------------
# 4. Per-ID rolling features (windows end the day BEFORE the death date)
# --------------------------------------------------------------------------
def build_id_features(daily: pd.DataFrame, train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    rolled = []
    for (lat, lon), grp in daily.groupby(["latitude", "longitude"]):
        grp = grp.sort_values("date").set_index("date")
        # reindex to a complete daily calendar so "7 days" always means 7 calendar days
        grp = grp.reindex(pd.date_range(grp.index.min(), grp.index.max(), freq="D"))
        out = pd.DataFrame(index=grp.index)
        for var in PARAMETERS:
            series = grp[var].shift(1)    # exclude the death date itself
            for w in WINDOWS:
                out[f"nasa_{var}_mean_{w}d"] = series.rolling(w, min_periods=max(1, w // 2)).mean()
            out[f"nasa_{var}_std_30d"] = series.rolling(30, min_periods=15).std()
        # accumulated rain over each window
        rain = grp["PRECTOTCORR"].shift(1)
        for w in WINDOWS:
            out[f"nasa_rain_sum_{w}d"] = rain.rolling(w, min_periods=max(1, w // 2)).sum()
        out["latitude"], out["longitude"] = lat, lon
        out.index.name = "deathdate"
        rolled.append(out.reset_index())
    lookup = pd.concat(rolled, ignore_index=True)

    people = pd.concat([train[["ID", "latitude", "longitude", "deathdate"]],
                        test[["ID", "latitude", "longitude", "deathdate"]]], ignore_index=True)
    people["deathdate"] = pd.to_datetime(people["deathdate"]).dt.normalize()
    features = people.merge(lookup, on=["latitude", "longitude", "deathdate"], how="left")
    return features.drop(columns=["latitude", "longitude", "deathdate"])


# --------------------------------------------------------------------------
# 5. Main
# --------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default="Data", help="folder containing Train.csv and Test.csv")
    parser.add_argument("--out-dir", default="nasa_output", help="where results are written")
    args, _unknown = parser.parse_known_args()   # ignores Jupyter/Colab's hidden '-f kernel.json'

    data_dir, out_dir = Path(args.data_dir), Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train, test = load_inputs(data_dir)
    locs, start, end = get_locations_and_dates(train, test)
    print(f"{len(locs)} unique locations | {start} -> {end} | "
          f"{len(locs) * (end.year - start.year + 1)} requests max\n", flush=True)

    failed = download_all(locs, start, end, out_dir / "cache")

    daily = consolidate(locs, start, end, out_dir / "cache")
    daily.to_csv(out_dir / "nasa_daily.csv", index=False)
    print(f"\nSaved nasa_daily.csv: {len(daily):,} rows", flush=True)

    features = build_id_features(daily, train, test)
    features.to_csv(out_dir / "nasa_features.csv", index=False)
    coverage = features.filter(like="nasa_").notna().all(axis=1).mean()
    print(f"Saved nasa_features.csv: {features.shape[0]:,} rows x {features.shape[1] - 1} features "
          f"| rows with complete features: {coverage:.1%}", flush=True)

    if failed or coverage < 0.95:
        print("\nSome data is missing - rerun the script to retry failed chunks before uploading.")
        return 1
    print("\nAll done. Upload nasa_output/nasa_features.csv back to Claude.")
    return 0


if __name__ == "__main__":
    code = main()
    if code and "ipykernel" not in sys.modules:   # only hard-exit from a real terminal
        sys.exit(code)

55 unique locations | 2007-06-29 -> 2022-12-13 | 880 requests max

[1/55] (0.5846, 33.5463)  chunks ok=16/880  failed=0
[2/55] (0.5496, 33.4903)  chunks ok=32/880  failed=0
[3/55] (0.5734, 33.4837)  chunks ok=48/880  failed=0
[4/55] (1.0723, 34.2266)  chunks ok=64/880  failed=0
[5/55] (0.5500, 33.4500)  chunks ok=80/880  failed=0
[6/55] (0.6085, 33.4755)  chunks ok=96/880  failed=0
[7/55] (0.5915, 33.5093)  chunks ok=112/880  failed=0
[8/55] (0.5264, 33.5198)  chunks ok=128/880  failed=0
[9/55] (0.6017, 33.4735)  chunks ok=144/880  failed=0
[10/55] (0.5645, 33.5067)  chunks ok=160/880  failed=0
[11/55] (0.7285, 33.4683)  chunks ok=176/880  failed=0
[12/55] (0.6135, 33.4841)  chunks ok=192/880  failed=0
[13/55] (0.5618, 33.5360)  chunks ok=208/880  failed=0
[14/55] (0.5944, 33.5202)  chunks ok=224/880  failed=0
[15/55] (0.5678, 33.5556)  chunks ok=240/880  failed=0
[16/55] (0.6084, 33.4827)  chunks ok=256/880  failed=0
[17/55] (0.5853, 33.4694)  chunks ok=272/880  failed=0
[18/55] (0.08

In [4]:
!pip install tabpfn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 30.1 MB/s eta 0:00:00


In [7]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

# Try importing TabPFN
TABPFN_AVAILABLE = False
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
    print("🚀 TabPFN successfully imported and ready!")
except ImportError:
    print("⚠️ TabPFN not found in environment. Ensemble will run with CatBoost, XGBoost, and LightGBM.")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)

# ==========================================
# 1. LOAD DATASETS
# ==========================================
def load_data():
    train_path = 'Train (6).csv'
    test_path = 'Test (7).csv'
    climate_path = 'climate_features.csv'
    nasa_path = 'nasa_features.csv'

    if not os.path.exists(train_path) and os.path.exists('/content/Train (6).csv'):
        train_path = '/content/Train (6).csv'
        test_path = '/content/Test (7).csv'
        climate_path = '/content/climate_features.csv'
        nasa_path = '/content/nasa_features.csv'

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    climate_feats = pd.read_csv(climate_path)
    nasa_feats = pd.read_csv(nasa_path)

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()
    if 'deathdate' in df.columns:
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year'] = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month
        df['dayofyear'] = df['deathdate'].dt.dayofyear

    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_range_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    if 'age' in df.columns:
        df['is_under5'] = (df['age'] < 5).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

exclude_cols = ['ID', 'deathdate', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_tabpfn = np.zeros(len(train)) if TABPFN_AVAILABLE else None

test_preds_cb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_tabpfn = np.zeros(len(test)) if TABPFN_AVAILABLE else None

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    # Fill missing values for tabular safety
    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # 1. CatBoost
    cb = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=5, eval_metric='AUC', random_seed=SEED + fold, verbose=0, auto_class_weights='Balanced')
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_preds_cb += cb.predict_proba(X_te)[:, 1] / 5.0

    # 2. XGBoost
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist')
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0, early_stopping_rounds=60)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 3. LightGBM
    lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(60, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 4. TabPFN (if available)
    if TABPFN_AVAILABLE:
        # TabPFN works best with scaled features and smaller sample sizes (subsample if needed)
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_te_scaled = scaler.transform(X_te)

        # TabPFN has sample size limits (~1000 rows), so we sample if train set is large
        if len(X_tr_scaled) > 1000:
            sub_idx = np.random.RandomState(SEED + fold).choice(len(X_tr_scaled), 1000, replace=False)
            X_tr_sub, y_tr_sub = X_tr_scaled[sub_idx], y_tr.iloc[sub_idx].values
        else:
            X_tr_sub, y_tr_sub = X_tr_scaled, y_tr.values

        tabpfn = TabPFNClassifier(device='cpu', N_ensemble_configurations=4)
        tabpfn.fit(X_tr_sub, y_tr_sub)
        oof_tabpfn[val_idx] = tabpfn.predict_proba(X_va_scaled)[:, 1]
        test_preds_tabpfn += tabpfn.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION
# ==========================================
if TABPFN_AVAILABLE:
    # Blend with TabPFN included
    blend_oof = 0.30 * oof_cb + 0.30 * oof_xgb + 0.20 * oof_lgb + 0.20 * oof_tabpfn
    blend_test = 0.30 * test_preds_cb + 0.30 * test_preds_xgb + 0.20 * test_preds_lgb + 0.20 * test_preds_tabpfn
else:
    # Standard 3-model blend
    blend_oof = 0.40 * oof_cb + 0.35 * oof_xgb + 0.25 * oof_lgb
    blend_test = 0.40 * test_preds_cb + 0.35 * test_preds_xgb + 0.25 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE
# ==========================================
sample_sub_path = 'SamplesSubmission (5).csv'
if not os.path.exists(sample_sub_path) and os.path.exists('/content/SamplesSubmission (5).csv'):
    sample_sub_path = '/content/SamplesSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)
sample_sub['is_climate_sensitive'] = blend_test

output_filename = 'submission_ensemble.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"📁 Submission file successfully saved to '{output_filename}'!")

🚀 TabPFN successfully imported and ready!
🌱 Global seed locked to 42


FileNotFoundError: [Errno 2] No such file or directory: 'nasa_features.csv'

In [6]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [11]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

# Try importing TabPFN
TABPFN_AVAILABLE = False
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
    print("🚀 TabPFN successfully imported and ready!")
except ImportError:
    print("⚠️ TabPFN not found in environment. Ensemble will run with CatBoost, XGBoost, and LightGBM.")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)

# ==========================================
# 1. LOAD DATASETS & CLEAN DATES IMMEDIATELY
# ==========================================
def load_data():
    train_path = 'Train (6).csv'
    test_path = 'Test (7).csv'
    climate_path = 'climate_features.csv'
    nasa_path = 'nasa_features.csv'

    if not os.path.exists(nasa_path) and os.path.exists('nasa_output/nasa_features.csv'):
        nasa_path = 'nasa_output/nasa_features.csv'

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    climate_feats = pd.read_csv(climate_path)
    nasa_feats = pd.read_csv(nasa_path)

    # Extract datetime features from train deathdate and drop string column
    if 'deathdate' in train.columns:
        dt = pd.to_datetime(train['deathdate'])
        train['year'] = dt.dt.year
        train['month'] = dt.dt.month
        train['dayofyear'] = dt.dt.dayofyear
        train = train.drop(columns=['deathdate'])

    # Extract datetime features from test deathdate and drop string column
    if 'deathdate' in test.columns:
        dt_test = pd.to_datetime(test['deathdate'])
        test['year'] = dt_test.dt.year
        test['month'] = dt_test.dt.month
        test['dayofyear'] = dt_test.dt.dayofyear
        test = test.drop(columns=['deathdate'])

    # Drop climate_features deathdate if present to avoid duplicate string columns
    if 'deathdate' in climate_feats.columns:
        climate_feats = climate_feats.drop(columns=['deathdate'])

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

# Merge external feature tables using ID
for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

# Drop any duplicate columns resulting from merges
train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

print(f"✅ Train shape after feature merge: {train.shape}")
print(f"✅ Test shape after feature merge: {test.shape}")

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_range_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    if 'age' in df.columns:
        df['is_under5'] = (df['age'] < 5).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

# Encode categorical columns
cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

# Exclude ID, target, and location (used for GroupKFold)
exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

# Final sanity check to ensure all features are numeric
for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        print(f"⚠️ Warning: Feature '{f}' is non-numeric ({train[f].dtype}) and will be dropped!")
        features.remove(f)

print(f"📊 Total active modeling features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_tabpfn = np.zeros(len(train)) if TABPFN_AVAILABLE else None

test_preds_cb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_tabpfn = np.zeros(len(test)) if TABPFN_AVAILABLE else None

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # 1. CatBoost
    cb = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=5, eval_metric='AUC', random_seed=SEED + fold, verbose=0, auto_class_weights='Balanced')
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_preds_cb += cb.predict_proba(X_te)[:, 1] / 5.0

    # 2. XGBoost
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist')
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0, early_stopping_rounds=60)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 3. LightGBM
    lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(60, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 4. TabPFN (if available)
    if TABPFN_AVAILABLE:
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_te_scaled = scaler.transform(X_te)

        if len(X_tr_scaled) > 1000:
            sub_idx = np.random.RandomState(SEED + fold).choice(len(X_tr_scaled), 1000, replace=False)
            X_tr_sub, y_tr_sub = X_tr_scaled[sub_idx], y_tr.iloc[sub_idx].values
        else:
            X_tr_sub, y_tr_sub = X_tr_scaled, y_tr.values

        tabpfn = TabPFNClassifier(device='cpu', N_ensemble_configurations=4)
        tabpfn.fit(X_tr_sub, y_tr_sub)
        oof_tabpfn[val_idx] = tabpfn.predict_proba(X_va_scaled)[:, 1]
        test_preds_tabpfn += tabpfn.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION
# ==========================================
if TABPFN_AVAILABLE:
    blend_oof = 0.30 * oof_cb + 0.30 * oof_xgb + 0.20 * oof_lgb + 0.20 * oof_tabpfn
    blend_test = 0.30 * test_preds_cb + 0.30 * test_preds_xgb + 0.20 * test_preds_lgb + 0.20 * test_preds_tabpfn
else:
    blend_oof = 0.40 * oof_cb + 0.35 * oof_xgb + 0.25 * oof_lgb
    blend_test = 0.40 * test_preds_cb + 0.35 * test_preds_xgb + 0.25 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE
# ==========================================
sample_sub_path = 'SamplesSubmission (5).csv'
sample_sub = pd.read_csv(sample_sub_path)
sample_sub['is_climate_sensitive'] = blend_test

output_filename = 'submission_ensemble.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"📁 Submission file successfully saved to '{output_filename}'!")

🚀 TabPFN successfully imported and ready!
🌱 Global seed locked to 42
✅ Train shape after feature merge: (3146, 62)
✅ Test shape after feature merge: (1030, 61)
📊 Total active modeling features: 66

--- Training Fold 1/5 ---


TypeError: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'

In [12]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

# Try importing TabPFN
TABPFN_AVAILABLE = False
try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
    print("🚀 TabPFN successfully imported and ready!")
except ImportError:
    print("⚠️ TabPFN not found in environment. Ensemble will run with CatBoost, XGBoost, and LightGBM.")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)

# ==========================================
# 1. LOAD DATASETS & CLEAN DATES IMMEDIATELY
# ==========================================
def load_data():
    train_path = 'Train (6).csv'
    test_path = 'Test (7).csv'
    climate_path = 'climate_features.csv'
    nasa_path = 'nasa_features.csv'

    if not os.path.exists(nasa_path) and os.path.exists('nasa_output/nasa_features.csv'):
        nasa_path = 'nasa_output/nasa_features.csv'

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    climate_feats = pd.read_csv(climate_path)
    nasa_feats = pd.read_csv(nasa_path)

    if 'deathdate' in train.columns:
        dt = pd.to_datetime(train['deathdate'])
        train['year'] = dt.dt.year
        train['month'] = dt.dt.month
        train['dayofyear'] = dt.dt.dayofyear
        train = train.drop(columns=['deathdate'])

    if 'deathdate' in test.columns:
        dt_test = pd.to_datetime(test['deathdate'])
        test['year'] = dt_test.dt.year
        test['month'] = dt_test.dt.month
        test['dayofyear'] = dt_test.dt.dayofyear
        test = test.drop(columns=['deathdate'])

    if 'deathdate' in climate_feats.columns:
        climate_feats = climate_feats.drop(columns=['deathdate'])

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

print(f"✅ Train shape after feature merge: {train.shape}")
print(f"✅ Test shape after feature merge: {test.shape}")

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_range_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    if 'age' in df.columns:
        df['is_under5'] = (df['age'] < 5).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Total active modeling features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_tabpfn = np.zeros(len(train)) if TABPFN_AVAILABLE else None

test_preds_cb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_tabpfn = np.zeros(len(test)) if TABPFN_AVAILABLE else None

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # 1. CatBoost
    cb = CatBoostClassifier(iterations=1200, learning_rate=0.03, depth=5, eval_metric='AUC', random_seed=SEED + fold, verbose=0, auto_class_weights='Balanced')
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_preds_cb += cb.predict_proba(X_te)[:, 1] / 5.0

    # 2. XGBoost (early_stopping_rounds passed in constructor for modern xgboost)
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=SEED + fold, eval_metric='auc', tree_method='hist',
        early_stopping_rounds=60
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 3. LightGBM
    lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(60, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 4. TabPFN (if available)
    if TABPFN_AVAILABLE:
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_va_scaled = scaler.transform(X_va)
        X_te_scaled = scaler.transform(X_te)

        if len(X_tr_scaled) > 1000:
            sub_idx = np.random.RandomState(SEED + fold).choice(len(X_tr_scaled), 1000, replace=False)
            X_tr_sub, y_tr_sub = X_tr_scaled[sub_idx], y_tr.iloc[sub_idx].values
        else:
            X_tr_sub, y_tr_sub = X_tr_scaled, y_tr.values

        tabpfn = TabPFNClassifier(device='cpu', N_ensemble_configurations=4)
        tabpfn.fit(X_tr_sub, y_tr_sub)
        oof_tabpfn[val_idx] = tabpfn.predict_proba(X_va_scaled)[:, 1]
        test_preds_tabpfn += tabpfn.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION
# ==========================================
if TABPFN_AVAILABLE:
    blend_oof = 0.30 * oof_cb + 0.30 * oof_xgb + 0.20 * oof_lgb + 0.20 * oof_tabpfn
    blend_test = 0.30 * test_preds_cb + 0.30 * test_preds_xgb + 0.20 * test_preds_lgb + 0.20 * test_preds_tabpfn
else:
    blend_oof = 0.40 * oof_cb + 0.35 * oof_xgb + 0.25 * oof_lgb
    blend_test = 0.40 * test_preds_cb + 0.35 * test_preds_xgb + 0.25 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE
# ==========================================
sample_sub_path = 'SamplesSubmission (5).csv'
sample_sub = pd.read_csv(sample_sub_path)
sample_sub['is_climate_sensitive'] = blend_test

output_filename = 'submission_ensemble.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"📁 Submission file successfully saved to '{output_filename}'!")

🚀 TabPFN successfully imported and ready!
🌱 Global seed locked to 42
✅ Train shape after feature merge: (3146, 62)
✅ Test shape after feature merge: (1030, 61)
📊 Total active modeling features: 66

--- Training Fold 1/5 ---


TypeError: TabPFNClassifier.__init__() got an unexpected keyword argument 'N_ensemble_configurations'

In [14]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting End-to-End Pipeline Execution in Colab...")

# ==========================================
# 1. LOAD DATASETS & CLEAN DATES IMMEDIATELY
# ==========================================
def load_data():
    train_path = 'Train (6).csv'
    test_path = 'Test (7).csv'
    climate_path = 'climate_features.csv'
    nasa_path = 'nasa_features.csv'

    if not os.path.exists(nasa_path) and os.path.exists('nasa_output/nasa_features.csv'):
        nasa_path = 'nasa_output/nasa_features.csv'

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    climate_feats = pd.read_csv(climate_path)
    nasa_feats = pd.read_csv(nasa_path)

    if 'deathdate' in train.columns:
        dt = pd.to_datetime(train['deathdate'])
        train['year'] = dt.dt.year
        train['month'] = dt.dt.month
        train['dayofyear'] = dt.dt.dayofyear
        train = train.drop(columns=['deathdate'])

    if 'deathdate' in test.columns:
        dt_test = pd.to_datetime(test['deathdate'])
        test['year'] = dt_test.dt.year
        test['month'] = dt_test.dt.month
        test['dayofyear'] = dt_test.dt.dayofyear
        test = test.drop(columns=['deathdate'])

    if 'deathdate' in climate_feats.columns:
        climate_feats = climate_feats.drop(columns=['deathdate'])

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

# Merge external feature tables using ID
for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

# Drop any duplicate columns resulting from merges
train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

print(f"✅ Train shape after feature merge: {train.shape}")
print(f"✅ Test shape after feature merge: {test.shape}")

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_range_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    if 'age' in df.columns:
        df['is_under5'] = (df['age'] < 5).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

# Encode categorical columns
cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

# Exclude ID, target, and location (used for GroupKFold)
exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

# Final sanity check to ensure all features are numeric
for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Total active modeling features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))

test_preds_cb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # 1. CatBoost
    cb = CatBoostClassifier(
        iterations=1200, learning_rate=0.03, depth=5,
        eval_metric='AUC', random_seed=SEED + fold, verbose=0,
        auto_class_weights='Balanced'
    )
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_preds_cb += cb.predict_proba(X_te)[:, 1] / 5.0

    # 2. XGBoost (with early_stopping_rounds in constructor for modern xgboost)
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=SEED + fold, eval_metric='auc', tree_method='hist',
        early_stopping_rounds=60
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 3. LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=SEED + fold, metric='auc', verbose=-1
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION (Fixed 0.5 Threshold)
# ==========================================
blend_oof = 0.40 * oof_cb + 0.35 * oof_xgb + 0.25 * oof_lgb
blend_test = 0.40 * test_preds_cb + 0.35 * test_preds_xgb + 0.25 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE SUBMISSION FILE
# ==========================================
sample_sub_path = 'SampleSubmission (5).csv'
if not os.path.exists(sample_sub_path) and os.path.exists('/content/SamplesSubmission (5).csv'):
    sample_sub_path = '/content/SamplesSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)
sample_sub['is_climate_sensitive'] = blend_test

output_filename = 'submission_ensemble.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"📁 Submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting End-to-End Pipeline Execution in Colab...
✅ Train shape after feature merge: (3146, 62)
✅ Test shape after feature merge: (1030, 61)
📊 Total active modeling features: 66

--- Training Fold 1/5 ---

--- Training Fold 2/5 ---

--- Training Fold 3/5 ---

--- Training Fold 4/5 ---

--- Training Fold 5/5 ---

🎉 ENSEMBLE TRAINING COMPLETED!
Out-of-Fold ROC-AUC: 0.81458
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.76289
Combined Score (0.6*F1 + 0.4*AUC): 0.78357
📁 Submission file successfully saved to 'submission_ensemble.csv'!


In [18]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting End-to-End Pipeline Execution in Colab...")

# ==========================================
# 1. LOAD DATASETS & CLEAN DATES IMMEDIATELY
# ==========================================
def load_data():
    train_path = 'Train (6).csv'
    test_path = 'Test (7).csv'
    climate_path = 'climate_features.csv'
    nasa_path = 'nasa_features.csv'

    if not os.path.exists(nasa_path) and os.path.exists('nasa_output/nasa_features.csv'):
        nasa_path = 'nasa_output/nasa_features.csv'

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    climate_feats = pd.read_csv(climate_path)
    nasa_feats = pd.read_csv(nasa_path)

    if 'deathdate' in train.columns:
        dt = pd.to_datetime(train['deathdate'])
        train['year'] = dt.dt.year
        train['month'] = dt.dt.month
        train['dayofyear'] = dt.dt.dayofyear
        train = train.drop(columns=['deathdate'])

    if 'deathdate' in test.columns:
        dt_test = pd.to_datetime(test['deathdate'])
        test['year'] = dt_test.dt.year
        test['month'] = dt_test.dt.month
        test['dayofyear'] = dt_test.dt.dayofyear
        test = test.drop(columns=['deathdate'])

    if 'deathdate' in climate_feats.columns:
        climate_feats = climate_feats.drop(columns=['deathdate'])

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

# Merge external feature tables using ID
for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

# Drop any duplicate columns resulting from merges
train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

print(f"✅ Train shape after feature merge: {train.shape}")
print(f"✅ Test shape after feature merge: {test.shape}")

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_range_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    if 'age' in df.columns:
        df['is_under5'] = (df['age'] < 5).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

# Encode categorical columns
cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

# Exclude ID, target, and location (used for GroupKFold)
exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

# Final sanity check to ensure all features are numeric
for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Total active modeling features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_cb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_lgb = np.zeros(len(train))

test_preds_cb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # 1. CatBoost
    cb = CatBoostClassifier(
        iterations=1200, learning_rate=0.03, depth=5,
        eval_metric='AUC', random_seed=SEED + fold, verbose=0,
        auto_class_weights='Balanced'
    )
    cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=60)
    oof_cb[val_idx] = cb.predict_proba(X_va)[:, 1]
    test_preds_cb += cb.predict_proba(X_te)[:, 1] / 5.0

    # 2. XGBoost
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=SEED + fold, eval_metric='auc', tree_method='hist',
        early_stopping_rounds=60
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 3. LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.03, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight,
        random_state=SEED + fold, metric='auc', verbose=-1
    )
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION (Fixed 0.5 Threshold)
# ==========================================
blend_oof = 0.40 * oof_cb + 0.35 * oof_xgb + 0.25 * oof_lgb
blend_test = 0.40 * test_preds_cb + 0.35 * test_preds_xgb + 0.25 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE CORRECT DUAL-TARGET SUBMISSION
# ==========================================
sample_sub_path = 'SampleSubmission (5).csv'
if not os.path.exists(sample_sub_path):
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    if sub_files:
        sample_sub_path = sub_files[0]

sample_sub = pd.read_csv(sample_sub_path)
print("📁 Loaded submission template columns:", sample_sub.columns.tolist())

# TargetF1 requires crisp binary decisions (0 or 1) at the 0.5 threshold
if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

# TargetRAUC requires ranked probabilities
if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_dual_target.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")
print("TargetF1 predictions breakdown:\n", sample_sub['TargetF1'].value_counts())

🌱 Global seed locked to 42
🚀 Starting End-to-End Pipeline Execution in Colab...
✅ Train shape after feature merge: (3146, 62)
✅ Test shape after feature merge: (1030, 61)
📊 Total active modeling features: 66

--- Training Fold 1/5 ---

--- Training Fold 2/5 ---

--- Training Fold 3/5 ---

--- Training Fold 4/5 ---

--- Training Fold 5/5 ---

🎉 ENSEMBLE TRAINING COMPLETED!
Out-of-Fold ROC-AUC: 0.81458
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.76289
Combined Score (0.6*F1 + 0.4*AUC): 0.78357
📁 Loaded submission template columns: ['ID', 'TargetF1', 'TargetRAUC']
🎉 Submission file successfully saved to 'submission_dual_target.csv'!
TargetF1 predictions breakdown:
 TargetF1
1    543
0    487
Name: count, dtype: int64


In [ ]:
# ==========================================
# 5. GENERATE SUBMISSION FILE
# ==========================================
# Update this string if your filename is slightly different (e.g., 'sampleSbmission.csv')
sample_sub_path = 'SampleSubmission (5).csv'

if not os.path.exists(sample_sub_path):
    # Fallback search for any csv with 'sub' in the name
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    if sub_files:
        sample_sub_path = sub_files[0]

sample_sub = pd.read_csv(sample_sub_path)
print("📁 Loaded submission template columns:", sample_sub.columns.tolist())

# Assign your ensemble predictions to the template target columns
if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = blend_test
if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_ensemble.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}' with shape {sample_sub.shape}!")

In [19]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting Advanced 0.87+ Pipeline Execution in Google Colab...")

# ==========================================
# 1. LOAD DATASETS (With Google Colab Path Support)
# ==========================================
def load_data():
    # Define primary local paths and Colab absolute paths
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key} in any expected path!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    # Clean deathdate strings immediately upon loading
    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

# Merge external feature tables using ID
for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

print(f"✅ Train shape after feature merge: {train.shape}")
print(f"✅ Test shape after feature merge: {test.shape}")

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Temperature spreads & heat stress proxies
    if 'tmax_30d' in df.columns and 'tmin_30d' in df.columns:
        df['temp_spread_30d'] = df['tmax_30d'] - df['tmin_30d']
    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    # Rain ratios & drought stress
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
            df['infant_heat_stress'] = df['is_infant'] * df['tavg_30d']
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

# Categorical Label Encoding
cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Total active modeling features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_rf = np.zeros(len(train))
oof_et = np.zeros(len(train))
oof_gb = np.zeros(len(train))

test_preds_rf = np.zeros(len(test))
test_preds_et = np.zeros(len(test))
test_preds_gb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")

    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    # Impute missing values
    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    # Scale features
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Random Forest Classifier
    rf = RandomForestClassifier(
        n_estimators=400, max_depth=12, min_samples_split=5,
        class_weight='balanced', random_state=SEED + fold, n_jobs=-1
    )
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Extra Trees Classifier
    et = ExtraTreesClassifier(
        n_estimators=500, max_depth=14, min_samples_split=4,
        class_weight='balanced', random_state=SEED + fold, n_jobs=-1
    )
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. Gradient Boosting Classifier
    gb = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.03, max_depth=5,
        subsample=0.8, random_state=SEED + fold
    )
    gb.fit(X_tr_scaled, y_tr)
    oof_gb[val_idx] = gb.predict_proba(X_va_scaled)[:, 1]
    test_preds_gb += gb.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION
# ==========================================
blend_oof = 0.35 * oof_rf + 0.45 * oof_et + 0.20 * oof_gb
blend_test = 0.35 * test_preds_rf + 0.45 * test_preds_et + 0.20 * test_preds_gb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 ADVANCED ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")
print(f"Combined Score (0.6*F1 + 0.4*AUC): {0.6 * final_f1 + 0.4 * final_auc:.5f}")

# ==========================================
# 5. GENERATE DUAL-TARGET SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    # Search directory for any submission template
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)
print(f"📁 Loaded submission template from '{sample_sub_path}' with columns: {sample_sub.columns.tolist()}")

# TargetF1 requires crisp binary decisions (0 or 1) at the 0.5 threshold
if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

# TargetRAUC requires ranked probabilities
if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_optimized_087.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")
print("TargetF1 predictions breakdown:\n", sample_sub['TargetF1'].value_counts())

🌱 Global seed locked to 42
🚀 Starting Advanced 0.87+ Pipeline Execution in Google Colab...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
✅ Train shape after feature merge: (3146, 65)
✅ Test shape after feature merge: (1030, 64)
📊 Total active modeling features: 73

--- Training Fold 1/5 ---

--- Training Fold 2/5 ---

--- Training Fold 3/5 ---

--- Training Fold 4/5 ---

--- Training Fold 5/5 ---

🎉 ADVANCED ENSEMBLE TRAINING COMPLETED!
Out-of-Fold ROC-AUC: 0.80850
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.78505
Combined Score (0.6*F1 + 0.4*AUC): 0.79443
📁 Loaded submission template from 'SampleSubmission (5).csv' with columns: ['ID', 'TargetF1', 'TargetRAUC']
🎉 Submission file successfully saved to 'submission_optimized_087.csv'!
TargetF1 predictions breakdown:
 TargetF1
1    668
0    362
Name: count, dtype: int64


In [20]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting High-Performance Pipeline Execution...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED FEATURE ENGINEERING & SMOOTHED TARGET ENCODING
# ==========================================
def engineer_features(df):
    df = df.copy()
    if 'tmax_30d' in df.columns and 'tmin_30d' in df.columns:
        df['temp_spread_30d'] = df['tmax_30d'] - df['tmin_30d']
    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']
            df['infant_heat_stress'] = df['is_infant'] * df['tavg_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

cat_cols = ['zone', 'gender', 'location']
for col in cat_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined_vals = pd.concat([train[col].astype(str), test[col].astype(str)], axis=0)
        le.fit(combined_vals)
        train[col] = le.transform(train[col].astype(str))
        test[col] = le.transform(test[col].astype(str))

exclude_cols = ['ID', 'is_climate_sensitive', 'location']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_rf = np.zeros(len(train))
oof_et = np.zeros(len(train))
oof_gb = np.zeros(len(train))

test_preds_rf = np.zeros(len(test))
test_preds_et = np.zeros(len(test))
test_preds_gb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    rf = RandomForestClassifier(n_estimators=500, max_depth=14, min_samples_split=4, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    et = ExtraTreesClassifier(n_estimators=600, max_depth=16, min_samples_split=3, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    gb = GradientBoostingClassifier(n_estimators=400, learning_rate=0.02, max_depth=5, subsample=0.8, random_state=SEED + fold)
    gb.fit(X_tr_scaled, y_tr)
    oof_gb[val_idx] = gb.predict_proba(X_va_scaled)[:, 1]
    test_preds_gb += gb.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. OPTIMAL THRESHOLD SEARCH FOR F1
# ==========================================
blend_oof = 0.30 * oof_rf + 0.50 * oof_et + 0.20 * oof_gb
blend_test = 0.30 * test_preds_rf + 0.50 * test_preds_et + 0.20 * test_preds_gb

best_thresh = 0.5
best_f1 = 0
for thresh in np.arange(0.30, 0.70, 0.01):
    score = f1_score(train[target_col], (blend_oof >= thresh).astype(int))
    if score > best_f1:
        best_f1 = score
        best_thresh = thresh

print(f"🎯 Optimal F1 Decision Threshold found: {best_thresh:.2f} (OOF F1: {best_f1:.5f})")
final_auc = roc_auc_score(train[target_col], blend_oof)
print(f"📊 Out-of-Fold ROC-AUC: {final_auc:.5f}")

# ==========================================
# 5. GENERATE OPTIMIZED SUBMISSION FILE
# ==========================================
sample_sub_path = 'SampleSubmission (5).csv'
if not os.path.exists(sample_sub_path):
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= best_thresh).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_optimized_thresh.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting High-Performance Pipeline Execution...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
🎯 Optimal F1 Decision Threshold found: 0.35 (OOF F1: 0.81114)
📊 Out-of-Fold ROC-AUC: 0.80741
🎉 Submission file successfully saved to 'submission_optimized_thresh.csv'!


In [21]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🛡️ Starting Private-LB-Safe Anti-Overfitting Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. SPATIALLY INVARIANT FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Climate anomalies & stress indicators
    if 'tmax_30d' in df.columns and 'tmin_30d' in df.columns:
        df['temp_spread_30d'] = df['tmax_30d'] - df['tmin_30d']
    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

# DROP all high-cardinality geographic and string identifiers that cause domain shift
exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Spatially-Invariant Features ({len(features)}): {features}")

# ==========================================
# 3. GROUPKFOLD WITH STABLE GENERALIZATION
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_lr = np.zeros(len(train))
oof_et = np.zeros(len(train))

test_preds_lr = np.zeros(len(test))
test_preds_et = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # Model 1: Highly Regularized Logistic Regression (Zero spatial overfitting)
    lr = LogisticRegression(C=0.01, class_weight='balanced', random_state=SEED + fold, max_iter=1000)
    lr.fit(X_tr_scaled, y_tr)
    oof_lr[val_idx] = lr.predict_proba(X_va_scaled)[:, 1]
    test_preds_lr += lr.predict_proba(X_te_scaled)[:, 1] / 5.0

    # Model 2: Heavily Regularized Extra Trees (Shallow depth to prevent memoization)
    et = ExtraTreesClassifier(n_estimators=800, max_depth=8, min_samples_split=15, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. STABLE BLENDING (Fixed 0.5 Threshold)
# ==========================================
blend_oof = 0.40 * oof_lr + 0.60 * oof_et
blend_test = 0.40 * test_preds_lr + 0.60 * test_preds_et

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🛡️ SAFE-GUARDED ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE SECURE SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_private_lb_safe.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Secure submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🛡️ Starting Private-LB-Safe Anti-Overfitting Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Spatially-Invariant Features (69): ['age', 'avg_temperature', 'max_temperature', 'min_temperature', 'precipitation', 'latitude', 'longitude', 'year', 'month', 'dayofyear', 'elevation', 'hot_days_30d', 'max_daily_rain_30d', 'ndvi_30d', 'ndvi_90d', 'rain_days_30d', 'rain_sum_30d', 'rain_sum_7d', 'rain_sum_90d', 'slope', 'tavg_30d', 'tavg_7d', 'tavg_90d', 'temp_range_mean_30d', 'tmax_30d', 'tmin_30d', 'year_dup', 'month_dup', 'dayofyear_dup', 'nasa_T2M_mean_7d', 'nasa_T2M_mean_30d', 'nasa_T2M_mean_90d', 'nasa_T2M_std_30d', 'nasa_T2M_MAX_mean_7d', 'nasa_T2M_MAX_mean_30d', 'nasa_T2M_MAX_mean_90d', 'nasa_T2M_MAX_std_30d', 'nasa_T2M_MIN_mean_7d', 'nasa_T2M_MIN_mean_30d', 'nasa_T2M_MIN_mean_90d', 'nasa_T2M_MIN_std_30d', 'nasa_PRECTOTCORR_mean_

In [22]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🌍 Starting Spatially-Enhanced NASA Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. SPATIAL & NASA FEATURE ENGINEERING
# ==========================================
def engineer_spatial_nasa_features(df):
    df = df.copy()

    # Ensure latitude and longitude are preserved and cleaned
    if 'latitude' in df.columns and 'longitude' in df.columns:
        # Spatial coordinate polynomials / interactions
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # NASA Meteorological & Climate Interactions
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])

    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']

    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    # Spatial-Climate Interactions (Mapping regional gradients smoothly)
    if 'latitude' in df.columns:
        if 'nasa_T2M_mean_30d' in df.columns:
            df['lat_temp_interaction'] = df['latitude'] * df['nasa_T2M_mean_30d']
        if 'nasa_rain_sum_30d' in df.columns:
            df['lat_rain_interaction'] = df['latitude'] * df['nasa_rain_sum_30d']

    if 'longitude' in df.columns:
        if 'nasa_T2M_mean_30d' in df.columns:
            df['lon_temp_interaction'] = df['longitude'] * df['nasa_T2M_mean_30d']
        if 'nasa_rain_sum_30d' in df.columns:
            df['lon_rain_interaction'] = df['longitude'] * df['nasa_rain_sum_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']

    return df

train = engineer_spatial_nasa_features(train)
test = engineer_spatial_nasa_features(test)

# Exclude only string identifiers (keeping lat, lon, and spatial features active!)
exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Spatially-Enhanced Features ({len(features)}): including Lat/Lon & NASA interactions.")

# ==========================================
# 3. GROUPKFOLD WITH SPATIAL-ROBUST ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_lr = np.zeros(len(train))
oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))

test_preds_lr = np.zeros(len(test))
test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Regularized Logistic Regression
    lr = LogisticRegression(C=0.01, class_weight='balanced', random_state=SEED + fold, max_iter=1000)
    lr.fit(X_tr_scaled, y_tr)
    oof_lr[val_idx] = lr.predict_proba(X_va_scaled)[:, 1]
    test_preds_lr += lr.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Regularized Extra Trees
    et = ExtraTreesClassifier(n_estimators=700, max_depth=10, min_samples_split=10, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. Regularized Random Forest
    rf = RandomForestClassifier(n_estimators=600, max_depth=10, min_samples_split=10, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. BLENDING & EVALUATION (Fixed 0.5 Threshold)
# ==========================================
blend_oof = 0.30 * oof_lr + 0.40 * oof_et + 0.30 * oof_rf
blend_test = 0.30 * test_preds_lr + 0.40 * test_preds_et + 0.30 * test_preds_rf

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🌍 SPATIAL-NASA ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_spatial_nasa.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🌍 Starting Spatially-Enhanced NASA Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Spatially-Enhanced Features (74): including Lat/Lon & NASA interactions.

🌍 SPATIAL-NASA ENSEMBLE COMPLETED!
Out-of-Fold ROC-AUC: 0.80949
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.76044
🎉 Submission file successfully saved to 'submission_spatial_nasa.csv'!


In [23]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting High-Performance Optimized Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED SPATIAL & NASA FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Spatial Coordinate Gradients
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # NASA Meteorological & Thermal Stress
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    # Spatial-Climate Interaction Gradients
    if 'latitude' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['lat_temp_interaction'] = df['latitude'] * df['nasa_T2M_mean_30d']
    if 'longitude' in df.columns and 'nasa_rain_sum_30d' in df.columns:
        df['lon_rain_interaction'] = df['longitude'] * df['nasa_rain_sum_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']
        if 'tavg_30d' in df.columns:
            df['under5_hot_stress'] = df['is_under5'] * df['tavg_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Modeling Features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & DIVERSE ENSEMBLE
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Extra Trees
    et = ExtraTreesClassifier(n_estimators=700, max_depth=14, min_samples_split=6, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=600, max_depth=14, min_samples_split=6, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. LightGBM
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    lgb_model = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 4. XGBoost
    xgb_model = xgb.XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist', early_stopping_rounds=50)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

# ==========================================
# 4. ENSEMBLE BLENDING & EVALUATION
# ==========================================
blend_oof = 0.30 * oof_et + 0.30 * oof_rf + 0.20 * oof_lgb + 0.20 * oof_xgb
blend_test = 0.30 * test_preds_et + 0.30 * test_preds_rf + 0.20 * test_preds_lgb + 0.20 * test_preds_xgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 HIGH-PERFORMANCE ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_high_performance.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting High-Performance Optimized Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Modeling Features: 73

--- Training Fold 1/5 ---

--- Training Fold 2/5 ---

--- Training Fold 3/5 ---

--- Training Fold 4/5 ---

--- Training Fold 5/5 ---

🎉 HIGH-PERFORMANCE ENSEMBLE COMPLETED!
Out-of-Fold ROC-AUC: 0.80845
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.78207
🎉 Submission file successfully saved to 'submission_high_performance.csv'!


In [24]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting 0.85+ Target Optimization Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED STRESS & SPATIAL ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Spatial Coordinate Gradients
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # Thermal & Evaporative Stress
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    # Extreme Temperature & Diurnal Spreads
    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']
            df['elderly_heat_index'] = df['is_elderly'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Modeling Features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & BLENDING
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))
oof_lr = np.zeros(len(train))
oof_lgb = np.zeros(len(train))

test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))
test_preds_lr = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Extra Trees
    et = ExtraTreesClassifier(n_estimators=800, max_depth=12, min_samples_split=8, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=700, max_depth=12, min_samples_split=8, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. Logistic Regression
    lr = LogisticRegression(C=0.05, class_weight='balanced', random_state=SEED + fold, max_iter=1000)
    lr.fit(X_tr_scaled, y_tr)
    oof_lr[val_idx] = lr.predict_proba(X_va_scaled)[:, 1]
    test_preds_lr += lr.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 4. LightGBM
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    lgb_model = lgb.LGBMClassifier(n_estimators=700, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

# ==========================================
# 4. CALIBRATED ENSEMBLE BLENDING
# ==========================================
blend_oof = 0.35 * oof_et + 0.35 * oof_rf + 0.15 * oof_lr + 0.15 * oof_lgb
blend_test = 0.35 * test_preds_et + 0.35 * test_preds_rf + 0.15 * test_preds_lr + 0.15 * test_preds_lgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 CALIBRATED ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_target_085.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting 0.85+ Target Optimization Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Modeling Features: 72

--- Training Fold 1/5 ---

--- Training Fold 2/5 ---

--- Training Fold 3/5 ---

--- Training Fold 4/5 ---

--- Training Fold 5/5 ---

🎉 CALIBRATED ENSEMBLE COMPLETED!
Out-of-Fold ROC-AUC: 0.81006
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.77703
🎉 Submission file successfully saved to 'submission_target_085.csv'!


In [25]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting Stacked Generalization Pipeline for 0.85+...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED STRESS & SPATIAL ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Spatial Coordinate Gradients
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # Thermal & Evaporative Stress
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']
            df['elderly_heat_index'] = df['is_elderly'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Modeling Features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD LEVEL-1 STACKING
# ==========================================
gkf = GroupKFold(n_splits=5)

# Level-1 Out-of-Fold (OOF) and Test prediction arrays
oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Level-1 Fold {fold + 1}/5 ---")
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Extra Trees
    et = ExtraTreesClassifier(n_estimators=900, max_depth=14, min_samples_split=6, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=800, max_depth=14, min_samples_split=6, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. LightGBM
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    lgb_model = lgb.LGBMClassifier(n_estimators=900, learning_rate=0.02, max_depth=5, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te)[:, 1] / 5.0

    # 4. XGBoost
    xgb_model = xgb.XGBClassifier(n_estimators=900, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist', early_stopping_rounds=50)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te)[:, 1] / 5.0

# ==========================================
# 4. LEVEL-2 STACKING META-LEARNER
# ==========================================
print("\n🔗 Training Level-2 Stacking Meta-Classifier...")
X_train_stack = np.column_stack([oof_et, oof_rf, oof_lgb, oof_xgb])
X_test_stack = np.column_stack([test_preds_et, test_preds_rf, test_preds_lgb, test_preds_xgb])

meta_learner = LogisticRegression(C=0.1, class_weight='balanced', random_state=SEED)
meta_learner.fit(X_train_stack, train[target_col])

blend_oof = meta_learner.predict_proba(X_train_stack)[:, 1]
blend_test = meta_learner.predict_proba(X_test_stack)[:, 1]

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 STACKED ENSEMBLE TRAINING COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_stacked_085.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Secure submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting Stacked Generalization Pipeline for 0.85+...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Modeling Features: 72

--- Training Level-1 Fold 1/5 ---

--- Training Level-1 Fold 2/5 ---

--- Training Level-1 Fold 3/5 ---

--- Training Level-1 Fold 4/5 ---

--- Training Level-1 Fold 5/5 ---

🔗 Training Level-2 Stacking Meta-Classifier...

🎉 STACKED ENSEMBLE TRAINING COMPLETED!
Out-of-Fold ROC-AUC: 0.81039
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.75594
🎉 Secure submission file successfully saved to 'submission_stacked_085.csv'!


In [26]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting 0.86+ Target Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED STRESS & SPATIAL ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Spatial Coordinate Gradients
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # Thermal & Evaporative Stress
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']
            df['elderly_heat_index'] = df['is_elderly'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Modeling Features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & BLENDING
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Extra Trees (Moderate depth for generalization)
    et = ExtraTreesClassifier(n_estimators=800, max_depth=10, min_samples_split=10, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=700, max_depth=10, min_samples_split=10, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. LightGBM
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    lgb_model = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 4. XGBoost
    xgb_model = xgb.XGBClassifier(n_estimators=800, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist', early_stopping_rounds=50)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. CALIBRATED BLENDING & FIXED 0.5 THRESHOLD
# ==========================================
blend_oof = 0.30 * oof_et + 0.30 * oof_rf + 0.20 * oof_lgb + 0.20 * oof_xgb
blend_test = 0.30 * test_preds_et + 0.30 * test_preds_rf + 0.20 * test_preds_lgb + 0.20 * test_preds_xgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 0.86+ OPTIMIZED ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_target_086.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Secure submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting 0.86+ Target Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Modeling Features: 72

--- Training Fold 1/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 2/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 3/5 ---


...


--- Training Fold 4/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 5/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



🎉 0.86+ OPTIMIZED ENSEMBLE COMPLETED!
Out-of-Fold ROC-AUC: 0.81376
Out-of-Fold F1-Score (at fixed 0.5 threshold): 0.77001
🎉 Secure submission file successfully saved to 'submission_target_086.csv'!


In [ ]:
# ==========================================
# 0. ENVIRONMENT & GLOBAL SEED SETUP
# ==========================================
import random
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    print(f"🌱 Global seed locked to {seed}")

seed_everything(SEED)
print("🚀 Starting Final 0.86+ Push Pipeline...")

# ==========================================
# 1. LOAD DATASETS (Google Colab Path Support)
# ==========================================
def load_data():
    paths = {
        'train': ['Train (6).csv', '/content/Train (6).csv'],
        'test': ['Test (7).csv', '/content/Test (7).csv'],
        'climate': ['climate_features.csv', '/content/climate_features.csv'],
        'nasa': ['nasa_features.csv', '/content/nasa_features.csv', 'nasa_output/nasa_features.csv', '/content/nasa_output/nasa_features.csv']
    }

    def find_file(key):
        for p in paths[key]:
            if os.path.exists(p):
                print(f"📁 Loaded {key} from: {p}")
                return pd.read_csv(p)
        raise FileNotFoundError(f"❌ Could not find file for {key}!")

    train = find_file('train')
    test = find_file('test')
    climate_feats = find_file('climate')
    nasa_feats = find_file('nasa')

    for df in [train, test, climate_feats]:
        if 'deathdate' in df.columns:
            dt = pd.to_datetime(df['deathdate'], errors='coerce')
            df['year'] = dt.dt.year
            df['month'] = dt.dt.month
            df['dayofyear'] = dt.dt.dayofyear
            df.drop(columns=['deathdate'], inplace=True, errors='ignore')

    return train, test, climate_feats, nasa_feats

train, test, climate_feats, nasa_feats = load_data()

for df_ref in [climate_feats, nasa_feats]:
    train = pd.merge(train, df_ref, on='ID', how='left', suffixes=('', '_dup'))
    test = pd.merge(test, df_ref, on='ID', how='left', suffixes=('', '_dup'))

train = train.loc[:, ~train.columns.duplicated()]
test = test.loc[:, ~test.columns.duplicated()]

# ==========================================
# 2. ADVANCED STRESS & SPATIAL ENGINEERING
# ==========================================
def engineer_features(df):
    df = df.copy()

    # Spatial Coordinate Gradients
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['coord_interaction'] = df['latitude'] * df['longitude']
        df['lat_sq'] = df['latitude'] ** 2
        df['lon_sq'] = df['longitude'] ** 2

    # Thermal, Evaporative & Drought Stress
    if 'nasa_T2M_mean_30d' in df.columns and 'nasa_RH2M_mean_30d' in df.columns:
        df['nasa_heat_index_30d'] = df['nasa_T2M_mean_30d'] + (0.33 * (df['nasa_RH2M_mean_30d'] / 100.0) * df['nasa_T2M_mean_30d'])
    if 'nasa_ALLSKY_SFC_SW_DWN_mean_30d' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['nasa_evap_demand_30d'] = df['nasa_ALLSKY_SFC_SW_DWN_mean_30d'] * df['nasa_T2M_mean_30d']
    if 'nasa_rain_sum_30d' in df.columns and 'nasa_rain_sum_90d' in df.columns:
        df['nasa_rain_intensity_ratio'] = df['nasa_rain_sum_30d'] / (df['nasa_rain_sum_90d'] / 3.0 + 1e-5)
        # Extreme drought/flood anomaly indicator
        df['nasa_drought_stress'] = df['nasa_rain_sum_30d'] * (df['nasa_T2M_mean_30d'] if 'nasa_T2M_mean_30d' in df.columns else 1.0)

    if 'nasa_T2M_MAX_mean_30d' in df.columns and 'nasa_T2M_MIN_mean_30d' in df.columns:
        df['nasa_diurnal_spread_30d'] = df['nasa_T2M_MAX_mean_30d'] - df['nasa_T2M_MIN_mean_30d']

    # Spatial-Climate Interaction Gradients
    if 'latitude' in df.columns and 'nasa_T2M_mean_30d' in df.columns:
        df['lat_temp_interaction'] = df['latitude'] * df['nasa_T2M_mean_30d']
    if 'longitude' in df.columns and 'nasa_rain_sum_30d' in df.columns:
        df['lon_rain_interaction'] = df['longitude'] * df['nasa_rain_sum_30d']

    # Vulnerability Age Interactions
    if 'age' in df.columns:
        df['is_infant'] = (df['age'] < 1).astype(int)
        df['is_under5'] = (df['age'] < 5).astype(int)
        df['is_elderly'] = (df['age'] >= 60).astype(int)
        if 'nasa_heat_index_30d' in df.columns:
            df['under5_heat_index'] = df['is_under5'] * df['nasa_heat_index_30d']
            df['elderly_heat_index'] = df['is_elderly'] * df['nasa_heat_index_30d']

    return df

train = engineer_features(train)
test = engineer_features(test)

exclude_cols = ['ID', 'is_climate_sensitive', 'location', 'zone', 'gender']
features = [c for c in train.columns if c not in exclude_cols]

for f in features:
    if not pd.api.types.is_numeric_dtype(train[f]):
        features.remove(f)

print(f"📊 Active Modeling Features: {len(features)}")

# ==========================================
# 3. GROUPKFOLD CROSS-VALIDATION & BLENDING
# ==========================================
gkf = GroupKFold(n_splits=5)

oof_et = np.zeros(len(train))
oof_rf = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

test_preds_et = np.zeros(len(test))
test_preds_rf = np.zeros(len(test))
test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))

target_col = 'is_climate_sensitive'
groups = train['location'].values if 'location' in train.columns else None

for fold, (train_idx, val_idx) in enumerate(gkf.split(train, train[target_col], groups)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_tr, y_tr = train.iloc[train_idx][features], train.iloc[train_idx][target_col]
    X_va, y_va = train.iloc[val_idx][features], train.iloc[val_idx][target_col]
    X_te = test[features]

    X_tr = X_tr.fillna(0)
    X_va = X_va.fillna(0)
    X_te = X_te.fillna(0)

    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_va_scaled = scaler.transform(X_va)
    X_te_scaled = scaler.transform(X_te)

    # 1. Extra Trees (Optimized depth for 0.86+ push)
    et = ExtraTreesClassifier(n_estimators=900, max_depth=12, min_samples_split=8, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    et.fit(X_tr_scaled, y_tr)
    oof_et[val_idx] = et.predict_proba(X_va_scaled)[:, 1]
    test_preds_et += et.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=800, max_depth=12, min_samples_split=8, class_weight='balanced', random_state=SEED + fold, n_jobs=-1)
    rf.fit(X_tr_scaled, y_tr)
    oof_rf[val_idx] = rf.predict_proba(X_va_scaled)[:, 1]
    test_preds_rf += rf.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 3. LightGBM
    pos_weight = (len(y_tr) - sum(y_tr)) / (sum(y_tr) + 1e-5)
    lgb_model = lgb.LGBMClassifier(n_estimators=900, learning_rate=0.02, max_depth=5, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, metric='auc', verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_preds_lgb += lgb_model.predict_proba(X_te_scaled)[:, 1] / 5.0

    # 4. XGBoost
    xgb_model = xgb.XGBClassifier(n_estimators=900, learning_rate=0.02, max_depth=4, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pos_weight, random_state=SEED + fold, eval_metric='auc', tree_method='hist', early_stopping_rounds=50)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_preds_xgb += xgb_model.predict_proba(X_te_scaled)[:, 1] / 5.0

# ==========================================
# 4. OPTIMIZED BLENDING (Emphasizing ET & LGB)
# ==========================================
blend_oof = 0.35 * oof_et + 0.25 * oof_rf + 0.25 * oof_lgb + 0.15 * oof_xgb
blend_test = 0.35 * test_preds_et + 0.25 * test_preds_rf + 0.25 * test_preds_lgb + 0.15 * test_preds_xgb

final_f1 = f1_score(train[target_col], (blend_oof >= 0.5).astype(int))
final_auc = roc_auc_score(train[target_col], blend_oof)

print("\n==========================================")
print("🎉 FINAL 0.86+ PUSH ENSEMBLE COMPLETED!")
print("==========================================")
print(f"Out-of-Fold ROC-AUC: {final_auc:.5f}")
print(f"Out-of-Fold F1-Score (at fixed 0.5 threshold): {final_f1:.5f}")

# ==========================================
# 5. GENERATE FINAL SUBMISSION FILE
# ==========================================
sample_sub_paths = ['SampleSubmission (5).csv', '/content/SampleSubmission (5).csv', 'samplesubmission (5).csv']
sample_sub_path = None
for p in sample_sub_paths:
    if os.path.exists(p):
        sample_sub_path = p
        break

if sample_sub_path is None:
    sub_files = [f for f in os.listdir('.') if 'sub' in f.lower() or 'sample' in f.lower()]
    sample_sub_path = sub_files[0] if sub_files else 'SampleSubmission (5).csv'

sample_sub = pd.read_csv(sample_sub_path)

if 'TargetF1' in sample_sub.columns:
    sample_sub['TargetF1'] = (blend_test >= 0.5).astype(int)

if 'TargetRAUC' in sample_sub.columns:
    sample_sub['TargetRAUC'] = blend_test

output_filename = 'submission_final_push_086.csv'
sample_sub.to_csv(output_filename, index=False)
print(f"🎉 Final secure submission file successfully saved to '{output_filename}'!")

🌱 Global seed locked to 42
🚀 Starting Final 0.86+ Push Pipeline...
📁 Loaded train from: Train (6).csv
📁 Loaded test from: Test (7).csv
📁 Loaded climate from: climate_features.csv
📁 Loaded nasa from: nasa_output/nasa_features.csv
📊 Active Modeling Features: 75

--- Training Fold 1/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 2/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 3/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 4/5 ---


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- Training Fold 5/5 ---
